# Vector stores and semantic search



In [10]:
from sentence_transformers import SentenceTransformer
import torch
import csv

## Part I: Basic vector store implementation

In [12]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.documents = []
        self.embeddings = None
        self.model = embedding_model

    def add_documents(self, documents: list[Document]):
        new_embeddings = self.model.encode(
            [doc.text for doc in documents],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)

        self.documents.extend(documents)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        embeded_query = self.model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
        scores = embeded_query @ self.embeddings.T

        sorted_scores, sorted_idx = torch.sort(scores, descending=True)
        top_scores = sorted_scores[:top_k]
        top_indices = sorted_idx[:top_k]

        return [SearchResult(score.item(), self.documents[idx]) for score, idx in zip(top_scores, top_indices)]

Dataset

In [13]:
documents = []

with open('data/animal-fun-facts-dataset.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for i, row in enumerate(reader):
        text = str(row.pop('text'))
        documents.append(Document(text=text, metadata=row))

print(f"Loaded {len(documents)} documents.\n")

for i, doc in enumerate(documents):
    if (i >= 5):
        break
    print(f"Document {i+1}:")
    print(doc.text)
    print(doc.metadata)
    print()

Loaded 7734 documents.

Document 1:
Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 2:
Aardvarks
have rather primitive brains that are very small for the size of the
animal. Some have suggested they are not particularly bright....
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 3:
Aardvarks
teeth are lined with fine upright tubes and have no roots or enamel.
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 4:
The aardvarks Latin family name "Tubulidentata" means "tube toothed"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-f

In [19]:
# Initialize the vector
model = SentenceTransformer('all-MiniLM-L6-v2')
vector_store = VectorStore(model)
vector_store.add_documents(documents)

# Queries
queries = [
    "Which animal can regenerate its limbs?",
    "How do whales communicate?",
    "What is the fastest land animal?",
    "Tell me a fun fact about elephants",
    "Which birds cannot fly?"
    ]

for query in queries:

    # Buscamos los 3 documentos más similares para cada pregunta
    results = vector_store.search(query, top_k=5)

    for res in results:
        print(f"Score: {res.score:.4f}")
        print(f"Text: {res.document.text}")
        print(f"Metadata: {res.document.metadata}")
        print("\n")
    print("-" *50)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13712.53it/s]


Score: 0.7367
Text: They can regenerate parts of their body! If they lose a limb it will grow back.
Metadata: {'animal_name': 'axolotl', 'source': '/r/AskReddit/comments/gbh7zz/what_are_some_really_amazing_animal_facts/fp67r9o/', 'media_link': '', 'wikipedia_link': '/wiki/Axolotl'}


Score: 0.7154
Text: Able to regrow lost or damaged limbs!
Metadata: {'animal_name': 'newt', 'source': 'https://a-z-animals.com/animals/newt/', 'media_link': '', 'wikipedia_link': '/wiki/Newt'}


Score: 0.6430
Text: They can regrow their arms.
This has been witnessed in one specimen from Newfoundland, in 1968, around the time when, for unknown reasons, these animals were washing up in droves.
Metadata: {'animal_name': 'giant squid', 'source': 'https://factanimal.com/giant-squid/', 'media_link': '', 'wikipedia_link': '/wiki/Giant_squid'}


Score: 0.6429
Text: Axolotl have an astonishing ability to regenerate body organs and lost limbs..
Incredibly, an Axolotl can grow back lost limbs in only a few weeks. It 

## Part II: Filtering by metadata

In [ ]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass